# Semana 2 · NumPy para tensores y álgebra lineal

**Nivel Intermedio** · Semana 2 · Clase expositiva [A] · 2 horas · CPU (no hace falta GPU)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/melvinpqbsc/Alto_Rendimiento_IA/blob/main/Nivel_Intermedio/clases/sem02_numpy.ipynb)

**Objetivos.** Al terminar la clase podrás:

1. Leer y cambiar la forma (*shape*) de un arreglo, y seleccionar partes con índices, cortes y máscaras.
2. Usar `axis` en las reducciones (`sum`, `mean`, `argmax`) y predecir la forma del resultado.
3. Aplicar las reglas de *broadcasting* para operar arreglos de formas distintas sin bucles.
4. Ver una capa de red neuronal como matriz × vector, y un lote (*batch*) como matriz × matriz.

**Prerrequisitos.** Semana 1: clases en Python y la capa `Linear` (aquí la repasamos). Lectura previa: *Introduction to NumPy* y *Linear Algebra Basics for ML* de MÓLÓ. La clase parte de ahí y llega hasta lo que PyTorch da por sabido: en la semana 8 verás que los tensores de PyTorch funcionan exactamente igual.

**Agenda.**

| Parte | Tema | Minutos |
|---|---|---|
| 1 | Por qué NumPy: bucles vs vectorización | 10 |
| 2 | Arreglos y formas | 15 |
| 3 | Indexación | 15 |
| 4 | Reducciones y `axis` | 15 |
| 5 | *Broadcasting* | 25 |
| 6 | Álgebra lineal: una capa y un lote | 25 |
| 7 | Cierre y tareas para la semana 3 | 5 |

Ejecuta esta celda primero: carga las librerías, fija la semilla y define `check`, la función que revisa tus ejercicios de "Tu turno".

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits

SEED = 0
rng = np.random.default_rng(SEED)


def check(name, test):
    """Ejecuta una prueba y muestra ✓, ✗ o "todavía sin resolver", sin detener el notebook."""
    try:
        test()
        print(f"✓ {name}")
    except NotImplementedError:
        print(f"… {name}: todavía sin resolver")
    except AssertionError as e:
        print(f"✗ {name}: {e}")
    except Exception as e:
        print(f"✗ {name}: {type(e).__name__}: {e}")


print("NumPy", np.__version__)

## 1. Por qué NumPy

Una red neuronal hace millones de multiplicaciones y sumas en cada paso de entrenamiento. Un bucle de Python hace una operación por vuelta y en cada vuelta pregunta qué tipo de dato tiene: es lento. NumPy guarda los números juntos en memoria, todos del mismo tipo, y hace el bucle en C. Escribir una operación sobre el arreglo entero en vez de un bucle se llama **vectorizar** (*vectorization*).

Comparemos con la misma cuenta, la suma de los cuadrados de un millón de números. El bucle de Python es la **línea base**: cualquier mejora se mide contra él.

In [ ]:
values = rng.normal(size=1_000_000)
values_list = values.tolist()

start = time.perf_counter()
total = 0.0
for v in values_list:
    total += v * v
t_loop = time.perf_counter() - start

start = time.perf_counter()
total_np = np.sum(values**2)
t_numpy = time.perf_counter() - start

print(f"bucle de Python: {t_loop * 1000:7.1f} ms  → {total:.4f}")
print(f"NumPy:           {t_numpy * 1000:7.1f} ms  → {total_np:.4f}")
print(f"NumPy es unas {t_loop / t_numpy:.0f} veces más rápido")

Mismo resultado, muchas veces más rápido. Por eso en *machine learning* casi nunca escribimos un bucle sobre los ejemplos: escribimos operaciones sobre arreglos enteros. El trabajo B1 es justamente eso: vectorizar funciones escritas con bucles.

## 2. Arreglos y formas

Un arreglo de NumPy (`np.ndarray`) tiene tres datos que conviene mirar siempre:

- `ndim`: cuántos ejes (*axes*) tiene.
- `shape`: cuántos elementos hay a lo largo de cada eje.
- `dtype`: el tipo de los números (`int64`, `float64`, …).

En *deep learning* a cualquier arreglo se le llama **tensor**. Los nombres según la cantidad de ejes:

| `ndim` | Nombre | Ejemplo | `shape` |
|---|---|---|---|
| 0 | escalar | una pérdida | `()` |
| 1 | vector | un ejemplo con 64 variables | `(64,)` |
| 2 | matriz | un lote de 32 ejemplos | `(32, 64)` |
| 3 | tensor de orden 3 | 32 imágenes en escala de grises de 8×8 | `(32, 8, 8)` |
| 4 | tensor de orden 4 | 32 imágenes a color de 224×224 (PyTorch) | `(32, 3, 224, 224)` |

In [ ]:
scalar = np.array(3.5)
vector = np.array([1, 2, 3])
matrix = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])

for name, a in [("scalar", scalar), ("vector", vector), ("matrix", matrix)]:
    print(f"{name:7} ndim={a.ndim}  shape={a.shape}  dtype={a.dtype}")

### Una imagen es un arreglo

Usamos todo el día el conjunto *digits* de sklearn: 1797 imágenes de dígitos escritos a mano, de 8×8 píxeles, con valores de 0 (blanco) a 16 (negro). Viene en dos formatos: `images`, con forma `(1797, 8, 8)`, y `data`, la misma información con cada imagen "aplanada" en una fila de 64 números.

In [ ]:
digits = load_digits()
images = digits.images  # (1797, 8, 8)
X = digits.data         # (1797, 64)
y = digits.target       # (1797,), el dígito de cada imagen

print("images:", images.shape, " X:", X.shape, " y:", y.shape)
print("la primera imagen, como números:")
print(images[0].astype(int))

fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for i, ax in enumerate(axes):
    ax.imshow(images[i], cmap="gray_r")
    ax.set_title(f"etiqueta {y[i]}")
    ax.axis("off")
plt.show()

### `reshape`: los mismos números, otra forma

`reshape` reorganiza los elementos sin copiarlos ni cambiarlos. La única condición: la cantidad total no cambia. Uno de los ejes puede ser `-1`, que significa "calcúlalo tú".

In [ ]:
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(2, -1).shape, " ← -1 se calculó como 6")
print(a.reshape(2, 3, 2).shape)

try:
    a.reshape(5, 3)
except ValueError as e:
    print("error:", e)

### Tu turno 1: formas

1. `images_from_X`: reconstruye `images` a partir de `X` con un `reshape`. Usa `-1` para el número de imágenes.
2. `as_column`: convierte `np.arange(5)`, de forma `(5,)`, en una columna de forma `(5, 1)`.
3. `first_flat`: la primera imagen de `images` aplanada en un vector de forma `(64,)`.

In [ ]:
images_from_X = None  # TODO
as_column = None      # TODO
first_flat = None     # TODO

In [ ]:
def test_shapes():
    if images_from_X is None or as_column is None or first_flat is None:
        raise NotImplementedError
    assert images_from_X.shape == (1797, 8, 8), f"forma {images_from_X.shape}, esperaba (1797, 8, 8)"
    assert np.array_equal(images_from_X, images), "los valores no coinciden con images"
    assert as_column.shape == (5, 1), f"forma {as_column.shape}, esperaba (5, 1)"
    assert first_flat.shape == (64,), f"forma {first_flat.shape}, esperaba (64,)"
    assert np.array_equal(first_flat, X[0]), "first_flat debe coincidir con X[0]"


check("Formas", test_shapes)

## 3. Indexación

Tres formas de elegir partes de un arreglo:

1. **Índices y cortes** (*slicing*), como en las listas, pero uno por eje y separados por comas: `M[fila, columna]`, `M[2:5, :]`.
2. **Máscaras booleanas** (*boolean masks*): un arreglo de `True`/`False` de la misma forma elige los elementos marcados con `True`.
3. **Indexación con listas** (*fancy indexing*): `X[[0, 5, 7]]` elige las filas 0, 5 y 7, en ese orden.

In [ ]:
M = np.arange(20).reshape(4, 5)
print(M)
print("M[1, 3]   =", M[1, 3])
print("M[1]      =", M[1], "  ← una fila entera")
print("M[:, 3]   =", M[:, 3], "  ← una columna entera")
print("M[1:3, ::2] =")
print(M[1:3, ::2])

Con imágenes, un corte en los ejes 1 y 2 es un recorte (*crop*). Una comparación como `y == 3` produce una máscara, y con ella elegimos todas las imágenes de un dígito.

In [ ]:
crop = images[0, 2:6, 2:6]   # imagen 0, filas 2–5, columnas 2–5
print("recorte:", crop.shape)

mask = y == 3
print("mask:", mask[:12], "…", mask.shape)
print("hay", mask.sum(), "imágenes del 3")  # True cuenta como 1

some = images[[0, 11, 22]]   # tres imágenes elegidas con una lista
print("some:", some.shape, "con etiquetas", y[[0, 11, 22]])

**Cuidado: un corte no es una copia.** Un corte es una *vista* (*view*): otra forma de mirar los mismos números en memoria. Si la modificas, cambia el original. Si necesitas una copia independiente, usa `.copy()`.

In [ ]:
a = np.arange(6)
view = a[:3]
view[0] = 100
print("a después de modificar el corte:", a)

a = np.arange(6)
copy = a[:3].copy()
copy[0] = 100
print("a después de modificar la copia:", a)

### Tu turno 2: elegir datos

1. `sevens`: todas las imágenes del dígito 7, con forma `(cantidad, 8, 8)`. Usa una máscara.
2. `n_sevens`: cuántas son, como entero.
3. `top_half`: la mitad superior (las primeras 4 filas de píxeles) de **todas** las imágenes, con forma `(1797, 4, 8)`.
4. `last_column`: la última columna de `X` (el píxel de abajo a la derecha de cada imagen), con forma `(1797,)`.

In [ ]:
sevens = None       # TODO
n_sevens = None     # TODO
top_half = None     # TODO
last_column = None  # TODO

In [ ]:
def test_indexing():
    if any(v is None for v in [sevens, n_sevens, top_half, last_column]):
        raise NotImplementedError
    assert sevens.shape == (179, 8, 8), f"sevens tiene forma {sevens.shape}"
    assert n_sevens == len(sevens), "n_sevens debe ser la cantidad de imágenes en sevens"
    assert top_half.shape == (1797, 4, 8), f"top_half tiene forma {top_half.shape}, esperaba (1797, 4, 8)"
    assert np.array_equal(top_half[5], images[5, :4]), "top_half[5] debe ser la mitad de arriba de la imagen 5"
    assert last_column.shape == (1797,), f"last_column tiene forma {last_column.shape}, esperaba (1797,)"
    assert np.array_equal(last_column, images[:, 7, 7]), "last_column debe ser el píxel de abajo a la derecha"


check("Indexación", test_indexing)

## 4. Reducciones y `axis`

`sum`, `mean`, `max`, `argmax` y compañía **reducen** un arreglo. Sin argumentos reducen todo a un número. Con `axis` reducen solo a lo largo de ese eje, y la regla para predecir la forma es una sola:

> **El eje que nombras desaparece.**

Para `images`, de forma `(1797, 8, 8)`:

| Expresión | Forma | Qué es |
|---|---|---|
| `images.mean()` | `()` | el brillo promedio de todo |
| `images.mean(axis=0)` | `(8, 8)` | la imagen promedio |
| `images.mean(axis=(1, 2))` | `(1797,)` | el brillo de cada imagen |
| `images.mean(axis=0, keepdims=True)` | `(1, 8, 8)` | igual que `axis=0`, pero el eje queda con tamaño 1 |

In [ ]:
print(images.mean().shape, images.mean(axis=0).shape, images.mean(axis=(1, 2)).shape)

# La imagen promedio de cada dígito: una máscara + una reducción en el eje 0.
fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for d, ax in enumerate(axes):
    ax.imshow(images[y == d].mean(axis=0), cmap="gray_r")
    ax.set_title(str(d))
    ax.axis("off")
plt.show()

`argmax` devuelve **la posición** del máximo, no el valor. Lo usaremos todo el año: una red produce un puntaje por clase y la predicción es `scores.argmax(axis=1)`.

In [ ]:
scores = np.array([[0.1, 2.0, 0.3],
                   [1.5, 0.2, 0.9]])
print("max por fila:   ", scores.max(axis=1))
print("argmax por fila:", scores.argmax(axis=1), " ← la clase predicha para cada ejemplo")

### Tu turno 3: reducciones

Trabaja con `X`, de forma `(1797, 64)`. Antes de escribir cada línea, di en voz alta qué forma debe tener el resultado.

1. `pixel_mean`: el valor promedio de cada uno de los 64 píxeles, con forma `(64,)`.
2. `brightness`: el brillo promedio de cada imagen, con forma `(1797,)`.
3. `darkest`: el **índice** de la imagen con mayor `brightness` (la que tiene más tinta).
4. `class_counts`: cuántas imágenes hay de cada dígito, con forma `(10,)`. Pista: `np.bincount`, o un bucle sobre los dígitos con una máscara.

In [ ]:
pixel_mean = None    # TODO
brightness = None    # TODO
darkest = None       # TODO
class_counts = None  # TODO

In [ ]:
def test_reductions():
    if any(v is None for v in [pixel_mean, brightness, darkest, class_counts]):
        raise NotImplementedError
    assert pixel_mean.shape == (64,), f"pixel_mean tiene forma {pixel_mean.shape}, esperaba (64,)"
    assert np.isclose(pixel_mean[0], 0.0) and np.isclose(pixel_mean[36], X[:, 36].mean()), "¿promediaste en el eje correcto?"
    assert brightness.shape == (1797,), f"brightness tiene forma {brightness.shape}, esperaba (1797,)"
    assert np.isclose(brightness[0], X[0].mean()), "brightness[0] debe ser el promedio de la imagen 0"
    assert brightness[darkest] == brightness.max(), "darkest debe ser la posición del máximo de brightness"
    assert np.shape(class_counts) == (10,) and class_counts.sum() == 1797, "class_counts: 10 conteos que suman 1797"
    assert class_counts[7] == 179, f"debería haber 179 sietes, class_counts[7] = {class_counts[7]}"


check("Reducciones", test_reductions)

## 5. *Broadcasting*

¿Qué pasa si sumamos arreglos de formas distintas? A veces NumPy "estira" el más chico para que encaje, sin copiar datos. Eso es el *broadcasting* (difusión). Ya lo usaste sin notarlo: en `values**2` o `x + 1`, el escalar se estira a la forma del arreglo.

**Las reglas.** Se comparan las formas **alineadas por la derecha**, eje por eje. Dos tamaños son compatibles si son iguales o si uno de ellos es 1 (el 1 se estira). Si a un arreglo le faltan ejes a la izquierda, se le agregan con tamaño 1.

| A | B | A + B |
|---|---|---|
| `(3, 4)` | `(4,)` | `(3, 4)`: la fila se suma a cada fila |
| `(3, 4)` | `(3, 1)` | `(3, 4)`: la columna se suma a cada columna |
| `(3, 1)` | `(1, 4)` | `(3, 4)`: los dos se estiran |
| `(3, 4)` | `(3,)` | error: 4 y 3 no son compatibles |

In [ ]:
A = np.zeros((3, 4))
row = np.array([1, 2, 3, 4])
col = np.array([[10], [20], [30]])

print("A + row:\n", A + row)
print("A + col:\n", A + col)
print("col + row:\n", col + row)

try:
    A + np.array([1, 2, 3])
except ValueError as e:
    print("error:", e)

Para agregar un eje de tamaño 1 donde haga falta se escribe `None` (o `np.newaxis`) en el índice: si `v` tiene forma `(3,)`, `v[:, None]` tiene forma `(3, 1)` y `v[None, :]` tiene forma `(1, 3)`. Así sale, por ejemplo, una tabla de multiplicar sin bucles:

In [ ]:
v = np.arange(1, 6)
print(v[:, None].shape, "×", v[None, :].shape, "→", (v[:, None] * v[None, :]).shape)
print(v[:, None] * v[None, :])

### Estandarizar: restar la media de cada columna

Antes de entrenar casi siempre **estandarizamos** (*standardize*) cada variable: le restamos su media y la dividimos por su desviación estándar, para que todas queden en una escala parecida. `X` tiene forma `(1797, 64)` y `X.mean(axis=0)` tiene forma `(64,)`: el *broadcasting* resta la media de cada columna a todas las filas.

Un detalle real: algunos píxeles valen 0 en todas las imágenes (los bordes), y su desviación estándar es 0. Dividir por 0 da `nan`, así que reemplazamos esas desviaciones por 1.

In [ ]:
mu = X.mean(axis=0)             # (64,)
sigma = X.std(axis=0)           # (64,)
print("píxeles con desviación 0:", (sigma == 0).sum())
sigma = np.where(sigma == 0, 1.0, sigma)

X_std = (X - mu) / sigma        # (1797, 64) - (64,) → (1797, 64)
print("forma:", X_std.shape)
print("media más alejada de 0:", np.abs(X_std.mean(axis=0)).max())
print("columnas con desviación 1:", np.isclose(X_std.std(axis=0), 1).sum(), "de 64 (las otras eran constantes)")

### Tu turno 4: todas las distancias de una vez

Para comparar imágenes, una medida natural es la **distancia euclidiana al cuadrado** entre dos vectores: la suma de las diferencias al cuadrado, píxel por píxel, $\lVert a - b \rVert^2 = \sum_k (a_k - b_k)^2$.

Completa `pairwise_sq_dists(A, B)`: recibe `A` de forma `(n, d)` y `B` de forma `(m, d)` y devuelve `D` de forma `(n, m)`, con `D[i, j]` = distancia al cuadrado entre `A[i]` y `B[j]`. **Sin bucles.** Plan:

1. `A[:, None, :]` tiene forma `(n, 1, d)` y `B[None, :, :]` tiene forma `(1, m, d)`.
2. Su resta, por *broadcasting*, tiene forma `(n, m, d)`: todas las diferencias.
3. Eleva al cuadrado y suma sobre el último eje.

In [ ]:
def pairwise_sq_dists(A, B):
    # TODO
    raise NotImplementedError

In [ ]:
def test_pairwise():
    A = rng.normal(size=(4, 3))
    B = rng.normal(size=(6, 3))
    D = pairwise_sq_dists(A, B)
    assert D.shape == (4, 6), f"D tiene forma {D.shape}, esperaba (4, 6)"
    reference = np.array([[np.sum((a - b) ** 2) for b in B] for a in A])  # la versión con bucles
    assert np.allclose(D, reference), "los valores no coinciden con la versión con bucles"


check("pairwise_sq_dists", test_pairwise)

### Un clasificador con lo que ya sabes

Con `pairwise_sq_dists` tenemos un clasificador completo: para cada imagen nueva, buscamos la imagen de entrenamiento **más parecida** (la de menor distancia) y copiamos su etiqueta. Se llama **vecino más cercano** (*nearest neighbor*) y lo verás a fondo en la semana 5.

Antes, la línea base: siempre predecir el dígito más común del entrenamiento. Esta celda funciona cuando completaste el Tu turno 4.

In [ ]:
perm = rng.permutation(len(X))      # los índices 0…1796 en orden aleatorio
train_idx, test_idx = perm[:1500], perm[1500:]
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

majority = np.bincount(y_train).argmax()
baseline_acc = np.mean(y_test == majority)
print(f"línea base (siempre {majority}): accuracy {baseline_acc:.3f}")

try:
    D = pairwise_sq_dists(X_test, X_train)  # (297, 1500)
    pred = y_train[D.argmin(axis=1)]        # la etiqueta del vecino más cercano
    print(f"vecino más cercano:        accuracy {np.mean(pred == y_test):.3f}")
except NotImplementedError:
    print("Completa el Tu turno 4 para ejecutar el vecino más cercano.")

Toda la predicción son tres líneas sin bucles: *broadcasting*, una reducción y una indexación con lista.

## 6. Álgebra lineal: una capa y un lote

### 6.1 El producto punto

El **producto punto** (*dot product*) multiplica dos vectores elemento a elemento y suma: $a \cdot b = \sum_k a_k b_k$. Es la operación más común de todo el *deep learning*: una neurona calcula $w \cdot x + b$, una suma de las entradas pesada por sus pesos. Primero con un bucle, después con NumPy:

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, -1.0, 0.5])

dot = 0.0
for k in range(len(a)):
    dot += a[k] * b[k]

print("bucle:      ", dot)
print("sum(a * b): ", np.sum(a * b))
print("a @ b:      ", a @ b)

### 6.2 Matriz × vector = una capa

Una capa con $k$ neuronas tiene $k$ vectores de pesos. Si los apilamos como filas de una matriz `W` de forma `(k, d)`, la capa entera es **un producto matriz × vector**: cada fila de `W` hace su producto punto con `x`.

$$W x = \begin{pmatrix} w_1 \cdot x \\ \vdots \\ w_k \cdot x \end{pmatrix}, \qquad (k, d) \; @ \; (d,) \to (k,)$$

Cada fila de `W` funciona como un **detector**: su producto punto es grande cuando `x` se parece a esa fila.

In [ ]:
W = rng.normal(size=(3, 4))   # una capa de 3 neuronas, cada una con 4 pesos
x = rng.normal(size=4)        # un ejemplo con 4 variables

by_rows = np.array([W[i] @ x for i in range(W.shape[0])])  # una neurona a la vez
print("fila por fila:", by_rows.round(3))
print("W @ x:        ", (W @ x).round(3), " forma", (W @ x).shape)

### 6.3 Matriz × matriz = un lote

Nunca pasamos un ejemplo a la vez: pasamos un **lote** (*batch*) de $n$ ejemplos, uno por fila de una matriz `X` de forma `(n, d)`. Toda la capa sobre todo el lote es un solo producto de matrices, más el sesgo por *broadcasting*:

$$Y = X W^\top + b, \qquad (n, d) \; @ \; (d, k) \to (n, k), \qquad (n, k) + (k,) \to (n, k)$$

Es la fórmula de la capa `Linear` de la semana 1, y la de `nn.Linear` en PyTorch. La regla de formas de `@`: **los dos ejes de adentro deben coincidir y desaparecen**, `(n, d) @ (d, k) → (n, k)`.

In [ ]:
X_batch = rng.normal(size=(5, 4))   # 5 ejemplos con 4 variables
b = np.array([0.1, 0.2, 0.3])       # un sesgo por neurona

Y = X_batch @ W.T + b
print("X:", X_batch.shape, "@ W.T:", W.T.shape, "+ b:", b.shape, "→ Y:", Y.shape)
print("la fila 2 de Y es la capa aplicada al ejemplo 2:", np.allclose(Y[2], W @ X_batch[2] + b))

### Tu turno 5: el producto de matrices por dentro

1. Completa `matmul_loops(A, B)` con bucles: `C[i, j]` es el producto punto de la fila `i` de `A` con la columna `j` de `B`. Si los ejes de adentro no coinciden, lanza `ValueError`. Puedes usar `@` para el producto punto de dos vectores, pero no entre matrices.
2. **Predice las formas** sin ejecutar nada: completa `predicted` con la forma del resultado de cada expresión (como tupla) o `"error"` si NumPy no puede calcularla. `a` tiene forma `(2, 3)`, `b` forma `(3, 4)`, `v` forma `(3,)` y `w` forma `(4,)`.

In [ ]:
def matmul_loops(A, B):
    # TODO
    raise NotImplementedError


predicted = {
    "a @ b": None,      # TODO
    "a @ v": None,      # TODO
    "b @ a": None,      # TODO
    "a.T @ a": None,    # TODO
    "a @ b + w": None,  # TODO
    "a @ b + v": None,  # TODO
}

In [ ]:
def test_matmul_loops():
    A = rng.normal(size=(3, 5))
    B = rng.normal(size=(5, 2))
    C = matmul_loops(A, B)
    assert C.shape == (3, 2), f"C tiene forma {C.shape}, esperaba (3, 2)"
    assert np.allclose(C, A @ B), "los valores no coinciden con A @ B"
    try:
        matmul_loops(A, A)
    except ValueError:
        pass
    else:
        raise AssertionError("(3, 5) @ (3, 5) debe lanzar ValueError")


def test_predicted_shapes():
    if any(v is None for v in predicted.values()):
        raise NotImplementedError
    env = {"a": np.ones((2, 3)), "b": np.ones((3, 4)), "v": np.ones(3), "w": np.ones(4)}
    wrong = []
    for expr, guess in predicted.items():
        try:
            actual = eval(expr, env).shape
        except ValueError:
            actual = "error"
        expected = guess if guess == "error" else tuple(guess)
        if expected != actual:
            wrong.append(expr)
    assert not wrong, f"revisa: {', '.join(wrong)}"


check("matmul_loops", test_matmul_loops)
check("Predecir formas", test_predicted_shapes)

¿Cuánto cuesta el bucle? El producto de dos matrices de 200×200 hace 8 millones de multiplicaciones:

In [ ]:
A = rng.normal(size=(200, 200))
B = rng.normal(size=(200, 200))
try:
    start = time.perf_counter()
    matmul_loops(A, B)
    t_loops = time.perf_counter() - start
    start = time.perf_counter()
    A @ B
    t_at = time.perf_counter() - start
    print(f"bucles: {t_loops * 1000:.0f} ms · A @ B: {t_at * 1000:.2f} ms · {t_loops / t_at:.0f} veces más rápido")
except NotImplementedError:
    print("Completa matmul_loops en el Tu turno 5 para ver la comparación.")

### 6.4 Todo junto: una capa que reconoce dígitos

Una capa lineal con pesos elegidos a mano, sin entrenar. Cada fila de `W` es un "detector" de un dígito: su imagen promedio. Ya vimos que el producto punto mide parecido, pero favorece a las imágenes con más tinta. La corrección viene de desarrollar la distancia al cuadrado a la imagen promedio $\mu_c$ del dígito $c$:

$$-\lVert x - \mu_c \rVert^2 = 2\,\mu_c \cdot x \;-\; \lVert \mu_c \rVert^2 \;-\; \lVert x \rVert^2$$

El último término es igual para todos los dígitos, así que no cambia cuál gana. Lo que queda es exactamente una capa lineal: pesos $W_c = 2\mu_c$ y sesgo $b_c = -\lVert \mu_c \rVert^2$. El dígito predicho es el de mayor puntaje: el promedio más cercano.

In [ ]:
means = np.stack([X_train[y_train == c].mean(axis=0) for c in range(10)])  # (10, 64)
W_digits = 2 * means                       # (10, 64): una fila por dígito
b_digits = -(means**2).sum(axis=1)         # (10,)

scores = X_test @ W_digits.T + b_digits    # (297, 64) @ (64, 10) + (10,) → (297, 10)
pred = scores.argmax(axis=1)
print("puntajes:", scores.shape)
print(f"línea base:                accuracy {baseline_acc:.3f}")
print(f"capa con pesos a mano:     accuracy {np.mean(pred == y_test):.3f}")

fig, axes = plt.subplots(1, 10, figsize=(12, 1.6))
for c, ax in enumerate(axes):
    ax.imshow(W_digits[c].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"fila {c}")
    ax.axis("off")
plt.show()

Una sola multiplicación de matrices clasifica 297 imágenes a la vez y deja muy atrás a la línea base. Los pesos los elegimos nosotros. **Entrenar** una red es dejar que el gradiente los elija: eso empieza en la semana 4.

## 7. Cierre

**Hoy vimos:**

- Vectorizar: operaciones sobre arreglos enteros en lugar de bucles, muchas veces más rápido.
- Formas: `ndim`, `shape`, `reshape` (con `-1`). Una imagen es un arreglo; un lote de imágenes, un tensor.
- Indexación: cortes (son vistas, no copias), máscaras booleanas e indexación con listas.
- Reducciones: el eje que nombras en `axis` desaparece. `argmax(axis=1)` convierte puntajes en predicciones.
- *Broadcasting*: formas alineadas por la derecha; un 1 se estira. `None` agrega un eje.
- Álgebra lineal: producto punto = una neurona; matriz × vector = una capa; matriz × matriz = un lote por esa capa. `(n, d) @ (d, k) → (n, k)`.

**Para la semana 3:**

1. **Entrega al inicio de la clase**: **B1: Ejercicios de NumPy** (vectorizar 6 funciones con bucles y reportar la aceleración), en tu repositorio de entregas, carpeta `B1/`.
2. **Lectura previa** (MÓLÓ, en inglés): [Advanced NumPy](https://colab.research.google.com/drive/1HWvN658jm7TuzNucOSTmFO95zzYfRs_1) (ayuda con el B1), [Data Analysis with Pandas](https://colab.research.google.com/drive/1JSY4kZpt-H5IUkEKturC-38-u9g4jIer) y [Descriptive statistics](https://colab.research.google.com/drive/12-WDEoD-CApbtgtVaACc60Z-EFVmOiju).
3. **Cuota semanal** en [la plataforma](https://ioai.artix.tech/).

La semana 3 es un laboratorio con datos reales en pandas.